In [2]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

customers = pd.read_csv('D:/Bharathi/python code/raw data/olist_customers_dataset.csv')
geolocation = pd.read_csv('D:/Bharathi/python code/raw data/olist_geolocation_dataset.csv')
order_items = pd.read_csv('D:/Bharathi/python code/raw data/olist_order_items_dataset.csv')
order_payments = pd.read_csv('D:/Bharathi/python code/raw data/olist_order_payments_dataset.csv')
reviews =pd.read_csv('D:/Bharathi/python code/raw data/olist_order_reviews_dataset.csv')
orders = pd.read_csv('D:/Bharathi/python code/raw data/olist_orders_dataset.csv')
sellers = pd.read_csv('D:/Bharathi/python code/raw data/olist_sellers_dataset.csv')
category_trans = pd.read_csv('D:/Bharathi/python code/raw data/product_category_name_translation.csv')
products =pd.read_csv('D:/Bharathi/python code/raw data/olist_products_dataset.csv')
#print(sellers_dataset.head(10))
print(" Loaded all 9 raw datasets!")

 Loaded all 9 raw datasets!


#### Cleaning the data and handling missing values ####

In [ ]:
print("--- 1. CLEANING ORDERS ---")
# Missing order_delivered_customer_date means the item was canceled or still in transit
# We do not drop them globally, but filter to 'delivered' orders for delivery analysis
date_cols = [
    'order_purchase_timestamp', 
    'order_approved_at', 
    'order_delivered_carrier_date', 
    'order_delivered_customer_date', 
    'order_estimated_delivery_date'
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

orders = orders.drop_duplicates(subset=['order_id'])

print(orders.isnull().sum())

print("--- 2. TRANSLATING & CLEANING PRODUCTS ---")  ### PRoducts dataset- missing product_category_name, which is MNAR (Missing Not At Random - user chose not to categorize)
# Merge products with English category translation
products = products.merge(category_trans, on='product_category_name', how='left')
# Fill missing category with 'unknown'
products['product_category_name_english'] = products['product_category_name_english'].fillna('unknown')
products.drop(columns=['product_category_name'], inplace=True)    #### Drop Portuguese column to avoid redundancy

# Median imputation for physical specs (dimensions/weights are right-skewed)
spec_cols = [
    'product_weight_g', 
    'product_length_cm', 
    'product_height_cm', 
    'product_width_cm', 
    'product_photos_qty',
    'product_name_lenght',
    'product_description_lenght']

for col in spec_cols:
    products[col] = products[col].fillna(products[col].median())

    
print("--- 3. CLEANING REVIEWS ---")  
### ORDER _REVIEWS dataset ####
# NaN review titles and comments are MNAR (Missing Not At Random - user chose not to write)

reviews['review_comment_title'] = reviews['review_comment_title'].fillna('No Title')
reviews['review_comment_message'] = reviews['review_comment_message'].fillna('No Comment')

### Convert review timestamps to datetime for analysis ###

reviews['review_creation_date'] = pd.to_datetime(reviews['review_creation_date'])
reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'])

# Deduplicate: if duplicate reviews exist for same review_id, keep latest answer
reviews = reviews.sort_values('review_answer_timestamp').drop_duplicates(subset=['review_id'], keep='last')

print("--- 4. CLEANING PAYMENTS & ORDER ITEMS ---")  ### order_payments dataset and order_items dataset

payments = order_payments.drop_duplicates()
# Outlier sanity: Payment value must be >= 0
payments = payments[payments['payment_value'] > 0]

order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'])
order_items = order_items.drop_duplicates()

print("\n--- 5. CLEANING CUSTOMERS, SELLERS & GEOLOCATION ---") ### Customer dataset, seller dataset and geolocation dataset

# Remove duplicate customer records based on primary identifier
customers = customers.drop_duplicates(subset=['customer_id'], keep='first')

# Remove duplicate seller records
sellers = sellers.drop_duplicates(subset=['seller_id'], keep='first')

# Geolocation: Aggregate duplicates per zip code to avoid combinatorial explosion
geolocation = geolocation.drop_duplicates()

##6. POST-CLEANING AUDIT (EDA Step 2.4 & Step 3.5 Verification)

print("\n--- POST-CLEANING INTEGRITY VERIFICATION ---")

# This is a Python Dictionary linking every cleaned DataFrame:
cleaned_datasets = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "order_payments": order_payments,
    "reviews": reviews,
    "sellers": sellers,
    "products": products,
    "geolocation": geolocation
}

# Verify nulls and duplicates across all tables
for name, df in cleaned_datasets.items():
    null_count = df.isnull().sum().sum()
    dup_count = df.duplicated().sum()
    print(f"Table: {name:15} | Rows: {df.shape[0]:<7} | Total Nulls: {null_count:<6} | Duplicate Rows: {dup_count}")

print("\nAll data are cleaned successfully.")

--- 1. CLEANING ORDERS ---
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64
--- 2. TRANSLATING & CLEANING PRODUCTS ---
--- 3. CLEANING REVIEWS ---
--- 4. CLEANING PAYMENTS & ORDER ITEMS ---

--- 5. CLEANING CUSTOMERS, SELLERS & GEOLOCATION ---

--- POST-CLEANING INTEGRITY VERIFICATION ---
Table: customers       | Rows: 99441   | Total Nulls: 0      | Duplicate Rows: 0
Table: orders          | Rows: 99441   | Total Nulls: 4908   | Duplicate Rows: 0
Table: order_items     | Rows: 112650  | Total Nulls: 0      | Duplicate Rows: 0
Table: order_payments  | Rows: 103886  | Total Nulls: 0      | Duplicate Rows: 0
Table: reviews         | Rows: 98410   | Total Nulls: 0      | Duplicate Rows: 0
Table: sellers         | Rows: 3095  